In [0]:
from datetime import datetime
from uuid import uuid4
from pyspark.sql import Row
import hashlib


CATALOGS = [
    "workspace"
]

SCAN_ID = str(uuid4())
SCAN_TIME = datetime.utcnow()

print(SCAN_ID)

In [0]:
tables=[]

for catalog in CATALOGS:

    schemas = spark.sql(
        f"SHOW SCHEMAS IN {catalog}"
    )
    for schema in schemas.collect():
        if schema.databaseName == 'default':
            print("entered")
            schema_name = schema.databaseName
            tbls = spark.sql(
                f"""
                SHOW TABLES 
                IN {catalog}.{schema_name}
                """
            )

            for t in tbls.collect():
                if not t.tableName.startswith("metadata_"):
                    tables.append(
                        {
                            "catalog": catalog,
                            "schema": schema_name,
                            "table": t.tableName
                        }
                    )


display(tables)

In [0]:

import hashlib

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    BooleanType,
    IntegerType,
)


# ============================================================
# 1. Generate ONE scan ID and ONE scan timestamp
#
# IMPORTANT:
# These must be created ONCE per complete metadata scan.
# Do NOT generate them inside the table loop.
# ============================================================

import uuid
from datetime import datetime, timezone

SCAN_ID = str(uuid.uuid4())
SCAN_TIME = datetime.now(timezone.utc)

print(f"[SCAN] scan_id={SCAN_ID}")
print(f"[SCAN] scan_time={SCAN_TIME}")


# ============================================================
# 2. Collect metadata
# ============================================================

metadata = []
errors = []


for t in tables:

    full_table = (
        f"`{t['catalog']}`."
        f"`{t['schema']}`."
        f"`{t['table']}`"
    )

    try:

        df = spark.table(full_table)

        # ----------------------------------------------------
        # Build schema hash
        # ----------------------------------------------------

        schema_string = "|".join(
            [
                f"{field.name}:"
                f"{field.dataType}:"
                f"{field.nullable}"
                for field in df.schema
            ]
        )

        schema_hash = hashlib.sha256(
            schema_string.encode("utf-8")
        ).hexdigest()

        # ----------------------------------------------------
        # One metadata row per column
        # ----------------------------------------------------

        for ordinal, field in enumerate(df.schema):

            metadata.append(
                Row(
                    scan_id=SCAN_ID,
                    scan_time=SCAN_TIME,

                    catalog_name=t["catalog"],
                    schema_name=t["schema"],
                    table_name=t["table"],

                    column_name=field.name,

                    data_type=str(field.dataType),

                    nullable=field.nullable,

                    row_count=None,

                    schema_hash=schema_hash,
                )
            )

    except Exception as e:

        errors.append(
            Row(
                scan_id=SCAN_ID,
                catalog_name=t["catalog"],
                schema_name=t["schema"],
                table_name=t["table"],
                error_message=str(e),
            )
        )

        print(
            f"[ERROR] {full_table}: {e}"
        )


# ============================================================
# 3. Define metadata schema
# ============================================================

metadata_schema = StructType(
    [
        StructField("scan_id", StringType(), False),
        StructField("scan_time", TimestampType(), False),

        StructField("catalog_name", StringType(), False),
        StructField("schema_name", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("column_name", StringType(), False),

        StructField("data_type", StringType(), False),
        StructField("nullable", BooleanType(), False),

        StructField("row_count", IntegerType(), True),
        StructField("schema_hash", StringType(), False),
    ]
)


# ============================================================
# 4. Create DataFrame
# ============================================================

metadata_df = spark.createDataFrame(
    metadata,
    schema=metadata_schema,
)


# ============================================================
# 5. SAFETY CHECK
#
# There must be only ONE row for each column inside a scan.
# ============================================================

duplicate_rows = (
    metadata_df
    .groupBy(
        "scan_id",
        "catalog_name",
        "schema_name",
        "table_name",
        "column_name",
    )
    .count()
    .filter("count > 1")
)

duplicate_count = duplicate_rows.count()

if duplicate_count > 0:

    print(
        "[ERROR] Duplicate metadata rows detected "
        "inside the current scan."
    )

    duplicate_rows.show(
        truncate=False
    )

    raise RuntimeError(
        "Metadata scan contains duplicate "
        "(scan_id, catalog, schema, table, column) keys."
    )


print(
    f"[SCAN] Metadata rows collected="
    f"{metadata_df.count()}"
)


# ============================================================
# 6. Write snapshot IDempotently
#
# DO NOT use blind append.
#
# A rerun of the same scan_id should update the existing
# snapshot instead of creating duplicate rows.
# ============================================================

TARGET_TABLE = "metadata_metadata_snapshot"


if not spark.catalog.tableExists(TARGET_TABLE):

    print(
        f"[SCAN] Creating {TARGET_TABLE}"
    )

    metadata_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(TARGET_TABLE)

else:

    target = DeltaTable.forName(
        spark,
        TARGET_TABLE
    )

    (
        target.alias("target")
        .merge(
            metadata_df.alias("source"),
            """
            target.scan_id = source.scan_id
            AND target.catalog_name = source.catalog_name
            AND target.schema_name = source.schema_name
            AND target.table_name = source.table_name
            AND target.column_name = source.column_name
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"[SCAN] Metadata snapshot merged into "
        f"{TARGET_TABLE}"
    )


# ============================================================
# 7. Optional: write scan errors
# ============================================================

if errors:

    errors_df = spark.createDataFrame(errors)

    errors_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "metadata_metadata_scan_errors"
        )

    print(
        f"[SCAN] {len(errors)} table errors recorded"
    )


In [0]:
scan_record = [
    Row(
        scan_id=SCAN_ID,
        scan_time=SCAN_TIME,
        status="SUCCESS",
        tables_scanned=len(tables),
        columns_scanned=len(metadata),
        error_count=len(errors)
    )
]

spark.createDataFrame(scan_record) \
    .write \
    .mode("append") \
    .saveAsTable(
        f"metadata_scan_history"
    )

In [0]:
%sql
select * from metadata_metadata_snapshot where table_name = 'disbursing_disbursement_vouchers' and scan_id = 'e67c912f-c91f-4901-9f14-bdc97a669d97'